# Colab OSS ingredient resolution pipeline

Runs the full `portion_pipeline_feasibility` pipeline on the canonical **1,000 recipes** (seed 42) using an open-source Hugging Face model instead of OpenAI.

**Prerequisites:** Colab GPU runtime (A100 recommended), Supabase credentials, AWS S3 access, and the input bundle at `s3://{artifacts}/colab/feasibility_1000_seed42/`.

See [`docs/COLAB_OSS_RESOLUTION.md`](../docs/COLAB_OSS_RESOLUTION.md) for setup.

## Progress

The pipeline cell shows **one tqdm bar** for all LLM prompts. `progress.json` under `OUT` updates every 10 prompts with live stats (and syncs to S3 when `COLAB_PROGRESS_S3_PREFIX` is set).

In [ ]:
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Use official Qwen3 14B model.
os.environ["OSS_MODEL_ID"] = "Qwen/Qwen3-14B"

# Keep Qwen3 thinking/reasoning enabled.
# This may improve judgment quality, but it can slow inference and use more output tokens. can be disabled with "0" if you want to speed up inference and save output tokens, but it may reduce judgment quality.
os.environ["QWEN_ENABLE_THINKING"] = "1"
#limit if you would like to run a small test first
os.environ["COLAB_LIMIT"] = "10"

# vLLM batching groups existing prompts together.
# This should improve throughput without creating extra prompts.
os.environ["VLLM_ENABLE_BATCHING"] = "1"
#tweak batch size if out of memory errors occur
os.environ["VLLM_BATCH_SIZE"] = "8"
os.environ["VLLM_BATCH_TIMEOUT_MS"] = "25"

In [ ]:
!nvidia-smi
!free -h

In [3]:
%%capture
!pip install -q transformers accelerate bitsandbytes sentence-transformers \
  pandas pyarrow psycopg2-binary rapidfuzz ingredient-parser-nlp \
  faiss-cpu scikit-learn tqdm psutil networkx python-dotenv boto3 nest_asyncio

In [4]:
%%capture
!pip uninstall -y vllm
!pip install -q vllm==0.23.0 \
  --extra-index-url https://wheels.vllm.ai/0.23.0/cu129 \
  --extra-index-url https://download.pytorch.org/whl/cu128

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from vllm import LLM
print('OK', torch.__version__, torch.version.cuda, torch.cuda.is_available())

In [ ]:
import os
from google.colab import userdata

def _secret(key: str, default: str = '') -> str:
    try:
        return userdata.get(key)
    except Exception:
        return os.environ.get(key, default)

for key in (
    'PG_POOL_USER', 'PG_PASSWORD', 'PG_POOL_HOST', 'PG_POOL_SESSION_PORT',
    'PG_DATABASE', 'PG_SSL_MODE', 'HF_TOKEN',
    'AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION',
    'S3_BUCKET_ARTIFACTS', 'S3_BUCKET_RAW',
):
    val = _secret(key)
    if val:
        os.environ[key] = val

os.environ.setdefault('PG_POOL_SESSION_PORT', '5432')
os.environ.setdefault('PG_DATABASE', 'postgres')
os.environ.setdefault('PG_SSL_MODE', 'require')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
os.environ.setdefault('OSS_MODEL_ID', 'Qwen/Qwen2.5-14B-Instruct')
print('Secrets loaded. S3 bucket:', os.environ.get('S3_BUCKET_ARTIFACTS'))

In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/Capstone')
if REPO.is_dir():
    !cd /content/Capstone && git fetch origin && git checkout agent_mvp && git pull origin agent_mvp
else:
    !git clone -b agent_mvp https://github.com/dcosta224/Capstone.git /content/Capstone
%cd /content/Capstone
sys.path.insert(0, str(REPO / 'scripts'))

In [ ]:
import os
from pathlib import Path

from colab_s3 import download_s3_file, sync_s3_prefix

CACHE = Path('/content/capstone_cache')
CACHE.mkdir(parents=True, exist_ok=True)
bucket = os.environ['S3_BUCKET_ARTIFACTS']
raw_bucket = os.environ.get('S3_BUCKET_RAW', '')

sync_s3_prefix(bucket, 'colab/feasibility_1000_seed42/', CACHE)

recipe_csv = CACHE / 'RecipeNLG.csv'
if not recipe_csv.is_file():
    if not raw_bucket:
        raise SystemExit('Set S3_BUCKET_RAW or place RecipeNLG.csv in the cache')
    download_s3_file(raw_bucket, 'Data/recipes/RecipeNLG.csv', recipe_csv)
print('Cache ready:', len(list(CACHE.iterdir())), 'items')

In [ ]:
import os
from pathlib import Path

src = Path('/content/capstone_cache/RecipeNLG.csv')
dst_dir = Path('/content/Capstone/Data/recipes')
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / 'RecipeNLG.csv'
if not src.is_file():
    raise SystemExit('RecipeNLG.csv missing from cache')
if dst.exists() or dst.is_symlink():
    dst.unlink()
os.symlink(src, dst)
print('Linked', dst, '->', src)

In [ ]:
import json
import os
import re
import threading
import asyncio
from dataclasses import dataclass, asdict
from typing import Any

import psutil
import torch

PROBE: dict[str, Any] = {}


def _gpu_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    props = torch.cuda.get_device_properties(0)
    return props.total_memory / (1024 ** 3)


def probe_memory() -> dict[str, Any]:
    """Set judge concurrency and embedding batch from GPU VRAM + system RAM."""
    vram = _gpu_vram_gb()
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

    cfg: dict[str, Any] = {
        "gpu_name": gpu_name,
        "vram_gb": round(vram, 1),
        "ram_gb": round(ram_gb, 1),
        "backend": "transformers",
        "judge_concurrency": 2,
        "vllm_max_num_seqs": 0,
        "max_new_tokens_judge": 512,
        "max_new_tokens_short": 256,
        "embed_batch": 0,
    }

    # Prefer vLLM on A100-class GPUs
    if vram >= 38:
        cfg["backend"] = "vllm"
        cfg["vllm_max_num_seqs"] = 8
        cfg["judge_concurrency"] = 8
        if vram >= 70:
            cfg["vllm_max_num_seqs"] = 12
            cfg["judge_concurrency"] = 12
    elif vram >= 14:
        cfg["backend"] = "transformers"
        cfg["judge_concurrency"] = 4
    else:
        cfg["backend"] = "transformers"
        cfg["judge_concurrency"] = 2

    if ram_gb >= 50:
        free_gb = psutil.virtual_memory().available / (1024 ** 3)
        cfg["embed_batch"] = min(512, max(128, int(free_gb * 48)))
    elif ram_gb >= 25:
        cfg["embed_batch"] = 256
    else:
        cfg["embed_batch"] = 0  # use pre-built S3 embeddings only

    min_parallel = int(os.environ.get("COLAB_MIN_BATCH", "8"))
    if cfg["backend"] == "vllm":
        cfg["vllm_max_num_seqs"] = max(min_parallel, cfg["vllm_max_num_seqs"])
        cfg["judge_concurrency"] = max(min_parallel, cfg["judge_concurrency"])
    elif vram >= 14:
        cfg["judge_concurrency"] = max(min_parallel, cfg["judge_concurrency"])
    elif min_parallel > cfg["judge_concurrency"]:
        print(
            f"WARNING: GPU VRAM {vram:.1f}GB — judge concurrency stays at "
            f"{cfg['judge_concurrency']} (set COLAB_MIN_BATCH lower or use A100 + vLLM for 8+)",
            flush=True,
        )

    return cfg


PROBE = probe_memory()
print(json.dumps(PROBE, indent=2))


def extract_json(text: str) -> dict[str, Any]:
    text = (text or "").strip()
    if not text:
        raise ValueError("empty model output")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        raise ValueError(f"no JSON object in: {text[:200]}")
    return json.loads(m.group(0))


@dataclass
class GenResult:
    parsed: dict[str, Any]
    raw: str
    prompt_tokens: int
    completion_tokens: int


@dataclass
class _BatchItem:
    prompt: str
    prompt_tokens: int
    max_new_tokens: int
    future: asyncio.Future


class OssJsonLlm:
    """Open-source JSON LLM for judge / enrichment / portion pick."""

    def __init__(self, model_id: str, probe: dict[str, Any]):
        self.model_id = model_id
        self.probe = probe
        self.backend = probe.get("backend", "transformers")
        self._lock = threading.Lock()
        self._llm = None
        self._model = None
        self._tokenizer = None
        self._load()

        # Async vLLM batching state. This batches concurrent judge/enrichment calls
        # without changing prompt text or max_new_tokens.
        self._batch_loop = None
        self._batch_lock = None
        self._batch_items = []
        self._batch_timer_task = None

    def _vllm_model_id(self) -> str:
        """
        Prefer AWQ weights for Qwen2.5 vLLM, but do not accidentally map Qwen3
        to a Qwen2.5 AWQ model.
        """
        if self.model_id.endswith("-AWQ"):
            return self.model_id

        # Qwen3 uses different official model IDs. Keep it as requested.
        if "Qwen3" in self.model_id or "qwen3" in self.model_id.lower():
            return self.model_id

        if "14B" in self.model_id:
            return "Qwen/Qwen2.5-14B-Instruct-AWQ"
        if "7B" in self.model_id:
            return "Qwen/Qwen2.5-7B-Instruct-AWQ"
        return self.model_id

    def _load(self) -> None:
        if self.backend == "vllm":
            try:
                from transformers import AutoTokenizer
                from vllm import LLM, SamplingParams  # noqa: F401

                max_seqs = int(self.probe.get("vllm_max_num_seqs", 8))
                vllm_id = self._vllm_model_id()

                # Load tokenizer once here instead of reloading it for every prompt.
                # Use self.model_id so prompt formatting stays aligned with the requested model.
                self._tokenizer = AutoTokenizer.from_pretrained(
                    self.model_id,
                    trust_remote_code=True,
                )

                self._llm = LLM(
                    model=vllm_id,
                    trust_remote_code=True,
                    gpu_memory_utilization=0.90,
                    max_model_len=8192,
                    max_num_seqs=max_seqs,
                    dtype="auto",
                )
                self._vllm_model_id_loaded = vllm_id
                self._SamplingParams = SamplingParams
                print(f"Loaded vLLM {vllm_id} max_num_seqs={max_seqs}")
                return
            except Exception as exc:
                print(f"vLLM load failed ({exc}); falling back to transformers")
                self.backend = "transformers"

        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        self._tokenizer = AutoTokenizer.from_pretrained(self.model_id, trust_remote_code=True)
        self._model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=bnb,
            device_map="auto",
            trust_remote_code=True,
        )
        self._model.eval()
        print(f"Loaded transformers 4-bit {self.model_id}")

    def _chat_prompt(self, system: str, user: str) -> str:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]

        kwargs = {
            "tokenize": False,
            "add_generation_prompt": True,
        }

        # Qwen3 can run in thinking mode. Disable it for this structured JSON task
        # to avoid extra reasoning text/tokens and slow JSON parsing.
        if "Qwen3" in self.model_id or "qwen3" in self.model_id.lower():
            kwargs["enable_thinking"] = (
                os.environ.get("QWEN_ENABLE_THINKING", "0").strip().lower()
                in {"1", "true", "yes", "on"}
            )

        try:
            return self._tokenizer.apply_chat_template(messages, **kwargs)
        except TypeError:
            # Older tokenizers may not accept enable_thinking.
            kwargs.pop("enable_thinking", None)
            return self._tokenizer.apply_chat_template(messages, **kwargs)

    def _count_tokens(self, text: str) -> int:
        if self._tokenizer is None:
            return 0
        return len(self._tokenizer.encode(text))

    def _build_prompt(
        self,
        system: str,
        user: str,
        extra_user: str | None = None,
    ) -> tuple[str, int]:
        full_user = user + (f"\n\n{extra_user}" if extra_user else "")
        full_user += "\n\nRespond with a single JSON object only. No markdown fences."
        prompt = self._chat_prompt(system, full_user)
        prompt_tokens = self._count_tokens(prompt)
        return prompt, prompt_tokens

    def _generate_one_from_prompt(self, prompt: str, prompt_tokens: int, max_new_tokens: int) -> GenResult:
        if self.backend == "vllm":
            params = self._SamplingParams(temperature=0.0, max_tokens=max_new_tokens)
            outs = self._llm.generate([prompt], params)
            raw = outs[0].outputs[0].text.strip()
            completion_tokens = self._count_tokens(raw)
            return GenResult(extract_json(raw), raw, prompt_tokens, completion_tokens)

        def _gen() -> str:
            inputs = self._tokenizer(prompt, return_tensors="pt").to(self._model.device)
            with torch.inference_mode():
                out = self._model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=self._tokenizer.eos_token_id,
                )
            new_tokens = out[0, inputs["input_ids"].shape[1] :]
            return self._tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        with self._lock:
            raw = _gen()
        completion_tokens = self._count_tokens(raw)
        return GenResult(extract_json(raw), raw, prompt_tokens, completion_tokens)

    def generate_json(
        self,
        system: str,
        user: str,
        *,
        max_new_tokens: int = 512,
        extra_user: str | None = None,
    ) -> GenResult:
        prompt, prompt_tokens = self._build_prompt(system, user, extra_user)
        return self._generate_one_from_prompt(prompt, prompt_tokens, max_new_tokens)

    def _ensure_batch_state(self) -> None:
        loop = asyncio.get_running_loop()
        if self._batch_loop is not loop:
            self._batch_loop = loop
            self._batch_lock = asyncio.Lock()
            self._batch_items = []
            self._batch_timer_task = None

    async def _delayed_flush_vllm_batch(self) -> None:
        timeout_ms = int(os.environ.get("VLLM_BATCH_TIMEOUT_MS", "25"))
        await asyncio.sleep(timeout_ms / 1000)

        async with self._batch_lock:
            items = self._batch_items
            self._batch_items = []
            self._batch_timer_task = None

        if items:
            await self._run_vllm_batch(items)

    async def _run_vllm_batch(self, items: list[_BatchItem]) -> None:
        # vLLM accepts one SamplingParams object per generate() call, so group by max_new_tokens.
        groups: dict[int, list[_BatchItem]] = {}
        for item in items:
            groups.setdefault(item.max_new_tokens, []).append(item)

        for max_new_tokens, group in groups.items():
            prompts = [item.prompt for item in group]
            params = self._SamplingParams(temperature=0.0, max_tokens=max_new_tokens)

            def _call_vllm():
                # Keep vLLM calls serialized, but each call may contain multiple prompts.
                with self._lock:
                    return self._llm.generate(prompts, params)

            try:
                outs = await asyncio.to_thread(_call_vllm)
                for item, out in zip(group, outs):
                    raw = out.outputs[0].text.strip()
                    try:
                        result = GenResult(
                            extract_json(raw),
                            raw,
                            item.prompt_tokens,
                            self._count_tokens(raw),
                        )
                        if not item.future.done():
                            item.future.set_result(result)
                    except Exception as exc:
                        if not item.future.done():
                            item.future.set_exception(exc)
            except Exception as exc:
                for item in group:
                    if not item.future.done():
                        item.future.set_exception(exc)

    async def _generate_json_async_batched_vllm(
        self,
        system: str,
        user: str,
        *,
        max_new_tokens: int = 512,
        extra_user: str | None = None,
    ) -> GenResult:
        self._ensure_batch_state()

        prompt, prompt_tokens = self._build_prompt(system, user, extra_user)
        loop = asyncio.get_running_loop()
        future = loop.create_future()
        item = _BatchItem(prompt, prompt_tokens, max_new_tokens, future)

        batch_size = int(os.environ.get("VLLM_BATCH_SIZE", "8"))

        async with self._batch_lock:
            self._batch_items.append(item)

            if len(self._batch_items) >= batch_size:
                items = self._batch_items
                self._batch_items = []
                self._batch_timer_task = None
                asyncio.create_task(self._run_vllm_batch(items))
            elif self._batch_timer_task is None or self._batch_timer_task.done():
                self._batch_timer_task = asyncio.create_task(self._delayed_flush_vllm_batch())

        return await future

    async def generate_json_async(self, *args, **kwargs) -> GenResult:
        if self.backend == "vllm" and os.environ.get("VLLM_ENABLE_BATCHING", "1") in {"1", "true", "yes", "on"}:
            return await self._generate_json_async_batched_vllm(*args, **kwargs)

        return await asyncio.to_thread(self.generate_json, *args, **kwargs)

    def meta(self) -> dict[str, Any]:
        meta = {
            "model_id": self.model_id,
            "backend": self.backend,
            "probe": self.probe,
            "vllm_batching": os.environ.get("VLLM_ENABLE_BATCHING", "1"),
            "vllm_batch_size": os.environ.get("VLLM_BATCH_SIZE", "8"),
            "qwen_enable_thinking": os.environ.get("QWEN_ENABLE_THINKING", "0"),
        }
        if hasattr(self, "_vllm_model_id_loaded"):
            meta["vllm_model_id"] = self._vllm_model_id_loaded
        return meta


DEFAULT_MODEL = os.environ.get("OSS_MODEL_ID", "Qwen/Qwen3-14B")
FALLBACK_MODEL = os.environ.get("OSS_FALLBACK_MODEL_ID", "Qwen/Qwen3-8B")

try:
    OSS_LLM = OssJsonLlm(DEFAULT_MODEL, PROBE)
except Exception as exc:
    print(f"OOM/loading failed for {DEFAULT_MODEL}: {exc}; trying {FALLBACK_MODEL}")
    min_parallel = int(os.environ.get("COLAB_MIN_BATCH", "8"))
    if PROBE.get("vram_gb", 0) >= 38:
        PROBE["judge_concurrency"] = max(min_parallel, PROBE.get("judge_concurrency", min_parallel))
        if PROBE.get("backend") == "vllm":
            PROBE["vllm_max_num_seqs"] = max(min_parallel, PROBE.get("vllm_max_num_seqs", min_parallel))
    else:
        PROBE["judge_concurrency"] = min(PROBE.get("judge_concurrency", 4), 4)
        print(
            "WARNING: fallback on small GPU — concurrency may be below COLAB_MIN_BATCH",
            flush=True,
        )
    OSS_LLM = OssJsonLlm(FALLBACK_MODEL, PROBE)

# Warmup: one short JSON call (validates backend without noisy multi-call output)
_r = OSS_LLM.generate_json(
    "You output JSON only.",
    'Return {"ok": true}',
    max_new_tokens=64,
)
print("OSS backend ready:", OSS_LLM.backend, OSS_LLM.model_id)

In [ ]:
import asyncio
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import ingredient_match_llm as iml
import line_enrichment_llm as lel
import portion_resolve_llm as prl
import openai_fallback
from portion_pipeline_feasibility import run_feasibility

CACHE = Path("/content/capstone_cache")
OUT = Path("/content/capstone_runs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT.mkdir(parents=True, exist_ok=True)
RUN_ID = OUT.name

os.environ["FOOD_4MACRO_CACHE"] = str(CACHE / "food_4macro.csv")
os.environ["CAPSTONE_RECIPE_CACHE"] = str(CACHE / "recipe_cache")

# Dummy OpenAI clients (unused after patches)
class _Dummy:
    class chat:
        class completions:
            @staticmethod
            async def create(**kwargs):
                raise RuntimeError("OpenAI client should not be called")


openai_fallback.get_async_openai_client = lambda: _Dummy()
openai_fallback.get_sync_openai_client = lambda: _Dummy()


async def oss_judge_async(
    client,
    model: str,
    user_prompt: str,
    valid_fdc_ids: set[int],
    *,
    system_prompt: str | None = None,
):
    sys_prompt = system_prompt or iml.SYSTEM_PROMPT
    prompt_tokens = completion_tokens = 0
    error = None
    parsed: dict = {}
    raw_response = None

    try:
        r = await OSS_LLM.generate_json_async(
            sys_prompt,
            user_prompt,
            max_new_tokens=PROBE.get("max_new_tokens_judge", 512),
        )
        parsed, raw_response = r.parsed, r.raw
        prompt_tokens += r.prompt_tokens
        completion_tokens += r.completion_tokens
        fdc_id = parsed.get("fdc_id")
        if fdc_id is not None and int(fdc_id) not in valid_fdc_ids:
            hint = (
                "Your previous fdc_id was not in the candidate list. "
                "Choose only from the listed fdc_id values, or null."
            )
            r2 = await OSS_LLM.generate_json_async(
                sys_prompt,
                user_prompt,
                max_new_tokens=PROBE.get("max_new_tokens_judge", 512),
                extra_user=hint,
            )
            parsed, raw_response = r2.parsed, r2.raw
            prompt_tokens += r2.prompt_tokens
            completion_tokens += r2.completion_tokens
            fdc_id = parsed.get("fdc_id")
            if fdc_id is not None and int(fdc_id) not in valid_fdc_ids:
                error = "invalid_fdc_id"
                parsed["fdc_id"] = None
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"

    matched_portion_id = parsed.get("matched_portion_id")
    if matched_portion_id is not None:
        try:
            matched_portion_id = int(matched_portion_id)
        except (TypeError, ValueError):
            matched_portion_id = None

    return {
        "fdc_id": parsed.get("fdc_id"),
        "certainty": parsed.get("certainty"),
        "rationale": parsed.get("rationale"),
        "matched_portion_id": matched_portion_id,
        "negligible_calories": bool(parsed.get("negligible_calories", False)),
        "response": raw_response,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "error": error,
    }


async def oss_enrich_one_async(client, model: str, ingredient: str, rules_plan):
    from line_enrichment_llm import build_user_prompt, validate_response

    prompt_tokens = completion_tokens = 0
    error = None
    parsed: dict = {}
    raw_response = None

    try:
        user = build_user_prompt(ingredient, rules_plan)
        r = await OSS_LLM.generate_json_async(
            lel.SYSTEM_PROMPT,
            user,
            max_new_tokens=PROBE.get("max_new_tokens_short", 256),
        )
        parsed, raw_response = r.parsed, r.raw
        prompt_tokens += r.prompt_tokens
        completion_tokens += r.completion_tokens
        validation_error = validate_response(parsed)
        if validation_error:
            r2 = await OSS_LLM.generate_json_async(
                lel.SYSTEM_PROMPT,
                user,
                max_new_tokens=PROBE.get("max_new_tokens_short", 256),
                extra_user=f"Validation failed ({validation_error}). Fix and resubmit.",
            )
            parsed, raw_response = r2.parsed, r2.raw
            prompt_tokens += r2.prompt_tokens
            completion_tokens += r2.completion_tokens
            if validate_response(parsed):
                error = validation_error
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"

    from ingredient_parse_llm import normalize_ingredient_key

    return {
        "ingredient_norm": normalize_ingredient_key(ingredient),
        "ingredient_raw": ingredient,
        "enrichment": parsed if not error else {},
        "certainty": parsed.get("certainty"),
        "rationale": parsed.get("rationale"),
        "error": error,
        "response": raw_response,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "price_estimate_usd": 0.0,
    }


def oss_pick_portion_sync(
    model: str,
    *,
    ingredient: str,
    quantity: float,
    unit: str | None,
    name: str | None,
    amount_kind: str,
    fdc_id: int,
    raw_rows: list,
):
    block = prl.format_portion_options(raw_rows)
    user = prl.build_user_prompt(
        ingredient=ingredient,
        quantity=quantity,
        unit=unit,
        name=name,
        amount_kind=amount_kind,
        fdc_id=fdc_id,
        portion_block=block,
    )
    r = OSS_LLM.generate_json(
        prl.SYSTEM_PROMPT,
        user,
        max_new_tokens=PROBE.get("max_new_tokens_short", 256),
    )
    parsed = r.parsed
    return {
        "portion_id": parsed.get("portion_id"),
        "certainty": parsed.get("certainty"),
        "rationale": parsed.get("rationale"),
        "prompt_tokens": r.prompt_tokens,
        "completion_tokens": r.completion_tokens,
        "price_estimate_usd": 0.0,
        "user_prompt": user,
    }


iml.judge_async = oss_judge_async
lel.enrich_one_async = oss_enrich_one_async
prl.pick_portion_sync = oss_pick_portion_sync

limit = int(os.environ["COLAB_LIMIT"]) if os.environ.get("COLAB_LIMIT") else None
model_label = OSS_LLM.model_id
min_parallel = int(os.environ.get("COLAB_MIN_BATCH", "8"))
judge_concurrency = max(min_parallel, int(PROBE.get("judge_concurrency", min_parallel)))

bucket = os.environ.get("S3_BUCKET_ARTIFACTS", "")
if bucket:
    os.environ["COLAB_PROGRESS_S3_PREFIX"] = f"s3://{bucket}/colab/runs/{RUN_ID}/"

recipe_csv = CACHE / "RecipeNLG.csv"

import nest_asyncio
nest_asyncio.apply()

report = run_feasibility(
    n_recipes=1000,
    seed=42,
    model=model_label,
    out_dir=OUT,
    baseline_dir=None,
    food_cache_dir=CACHE / "food_cache",
    limit=limit,
    concurrency=judge_concurrency,
    skip_portion_llm=False,
    force_amount=False,
    force_judging=False,
    force_payloads=False,
    force_all=False,
    finalize_only=False,
    only_no_portion=False,
    use_mlflow=False,
    progress_mode="colab",
    disk_flush_every=10,
    judge_log_every=10_000,
    parquet_compact_every=10,
    enrichment_concurrency=judge_concurrency,
    sample_manifest=CACHE / "sampled_recipe_ids.json",
    recipe_csv=recipe_csv,
    recipe_cache_dir=CACHE / "recipe_cache",
)

(OUT / "oss_model_meta.json").write_text(
    json.dumps({**OSS_LLM.meta(), "run_id": RUN_ID, "colab_limit": limit}, indent=2) + "\n"
)
print("Run complete:", OUT)
print(json.dumps({k: report[k] for k in (
    "n_lines", "fdc_match_rate_all", "gram_resolve_rate_all",
    "fdc_and_gram_rate_all", "judge_error_count", "elapsed_sec",
) if k in report}, indent=2))


In [ ]:
from pathlib import Path

from colab_s3 import upload_dir_to_s3

bucket = os.environ['S3_BUCKET_ARTIFACTS']
prefix = f'colab/runs/{RUN_ID}/'
upload_dir_to_s3(OUT, bucket, prefix)
print('Uploaded to', f's3://{bucket}/{prefix}')

In [ ]:
import json
from pathlib import Path

baseline = json.loads((CACHE / 'baseline_summary.json').read_text())
new_report = json.loads((OUT / 'feasibility_report.json').read_text())

keys = [
    'fdc_match_rate_all', 'gram_resolve_rate_all', 'fdc_and_gram_rate_all',
    'fdc_and_gram_rate_needs_portion', 'judge_error_count', 'n_lines',
]
print('metric | baseline (GPT) | OSS')
for k in keys:
    b = baseline.get(k)
    n = new_report.get(k)
    print(f'{k}: {b} -> {n}')